<a href="https://colab.research.google.com/github/gitanisa008/palm_oil_detection/blob/main/train_palm_yolov8.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Palm Oil Detection - YOLOv8
**Input:** ArcGIS Image Chips (KITTI format)  
**Output:** Model `.onnx dan .emd` untuk deteksi pohon sawit

---
### Sebelum mulai:
1. Runtime → Change runtime type → **GPU (T4)**
2. Upload folder export ArcGIS ke Google Drive
3. Jalankan cell satu per satu dari atas ke bawah

## Step 1: Install YOLOv8

In [ ]:
!pip install ultralytics -q
from ultralytics import YOLO
print('YOLOv8 ready!')

## Step 2: Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
print(os.listdir('/content/drive/MyDrive'))

## Step 3: Set Path
Ganti sesuai nama folder export kamu di Google Drive

In [ ]:

TRAINING_PATH = '/content/drive/MyDrive/image_chips'
OUTPUT_PATH   = '/content/drive/MyDrive/palm_yolo_output'

print('Training data:', TRAINING_PATH)
print('Output model :', OUTPUT_PATH)

## Step 4: Cek GPU

In [ ]:
import torch
print('GPU tersedia :', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU          :', torch.cuda.get_device_name(0))
    print('VRAM         :', round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 1), 'GB')
else:
    print('WARNING: Ganti runtime ke GPU dulu!')

## Step 5: Cek Ukuran Image Chips

In [ ]:
from PIL import Image

images = [f for f in os.listdir(f'{TRAINING_PATH}/images')
          if f.endswith(('.jpg', '.png', '.tif', '.tiff'))]

sample = images[0]
img = Image.open(f'{TRAINING_PATH}/images/{sample}')

IMG_W, IMG_H = img.size
print('Ukuran chip  :', img.size)
print('Contoh file  :', sample)
print('Total images :', len(images))

## Step 6: Konversi KITTI → YOLO

In [ ]:
import shutil

os.makedirs('dataset/images/train', exist_ok=True)
os.makedirs('dataset/labels/train', exist_ok=True)

# Copy images (skip .tfw dll)
for f in os.listdir(f'{TRAINING_PATH}/images'):
    if f.endswith(('.jpg', '.png', '.tif', '.tiff')):
        shutil.copy(f'{TRAINING_PATH}/images/{f}', 'dataset/images/train/')

# Konversi label KITTI → YOLO
skipped = 0
converted = 0
for txt in os.listdir(f'{TRAINING_PATH}/labels'):
    if not txt.endswith('.txt'):
        continue
    with open(f'{TRAINING_PATH}/labels/{txt}') as f:
        lines = f.readlines()
    yolo_lines = []
    for line in lines:
        parts = line.strip().split()
        if len(parts) < 8:
            skipped += 1
            continue
        cls = 0  # palm = kelas 0
        x1, y1, x2, y2 = float(parts[4]), float(parts[5]), float(parts[6]), float(parts[7])
        cx = ((x1 + x2) / 2) / IMG_W
        cy = ((y1 + y2) / 2) / IMG_H
        w  = (x2 - x1) / IMG_W
        h  = (y2 - y1) / IMG_H
        yolo_lines.append(f'{cls} {cx:.6f} {cy:.6f} {w:.6f} {h:.6f}')
    with open(f'dataset/labels/train/{txt}', 'w') as f:
        f.write('\n'.join(yolo_lines))
    converted += 1

print('Konversi selesai!')
print('Total images :', len(os.listdir('dataset/images/train')))
print('Total labels :', converted)
print('Baris skipped:', skipped)

## Step 7: Buat Konfigurasi Dataset (palm.yaml)

In [ ]:
yaml_content = f"""path: /content/dataset
train: images/train
val: images/train

nc: 1
names:
  0: palm
"""

with open('palm.yaml', 'w') as f:
    f.write(yaml_content)

print('palm.yaml siap!')
print(yaml_content)

## Step 8: Training

In [ ]:
from ultralytics import YOLO

model = YOLO('yolov8n.pt')  

results = model.train(
    data='palm.yaml',
    epochs=100,
    imgsz=256,
    batch=16,
    project=OUTPUT_PATH,
    name='palm_detection',
    save=True
)

print('Training selesai!')
print('Model tersimpan di:', OUTPUT_PATH)

## Step 9: Evaluasi Hasil dan Ekspor Model .onnx dan .emd

In [ ]:
# Load model terbaik hasil training
best_model = YOLO(f'{OUTPUT_PATH}/palm_detection/weights/best.pt')

# Validasi
metrics = best_model.val(data='palm.yaml')
print('mAP50   :', round(metrics.box.map50, 4))
print('mAP50-95:', round(metrics.box.map, 4))

In [ ]:
# Visualisasi prediksi 
import glob
from IPython.display import Image as IPImage, display

sample_imgs = glob.glob('dataset/images/train/*.tif')[:3]

for img_path in sample_imgs:
    result = best_model.predict(img_path, conf=0.3, save=True,
                                project='/content/predictions')

# Tampilkan hasil
pred_imgs = glob.glob('/content/predictions/predict/*.jpg')[:3]
for p in pred_imgs:
    display(IPImage(p))

In [ ]:
from ultralytics import YOLO

model = YOLO(f'{OUTPUT_PATH}/palm_detection/weights/best.pt')

# Export ke format ONNX
model.export(format='onnx')

In [ ]:
import json

emd = {
    "Framework": "PyTorch",
    "ModelConfiguration": "FastaiSSD",
    "ModelFile": "best.onnx",
    "ModelType": "ObjectDetection",
    "ImageWidth": 256,
    "ImageHeight": 256,
    "ImageSpaceUsed": "MAP_SPACE",
    "NormalizationStats": {
        "band_min_values": [0, 0, 0],
        "band_max_values": [255, 255, 255],
        "band_mean_values": [0, 0, 0],
        "band_std_values": [1, 1, 1],
        "scaled_min_values": [0, 0, 0],
        "scaled_max_values": [1, 1, 1]
    },
    "Classes": [
        {"Value": 0, "Name": "palm", "Color": [255, 0, 0]}
    ]
}

with open(f'{OUTPUT_PATH}/palm_detection/weights/best.emd', 'w') as f:
    json.dump(emd, f, indent=2)

print('Done!')

# Step 10: Running Deteksi & Ekspor SHP

In [ ]:
import geopandas as gpd
from shapely.geometry import box
from ultralytics import YOLO
import rasterio
from rasterio.windows import Window
import numpy as np
import torch

# ── PILIH MODEL ────────────────────────────────────────────────────────────────
# Opsi 1 — pakai hasil training dari notebook ini (default)
# MODEL_PATH = f'{OUTPUT_PATH}/palm_detection/weights/best.pt'

# Opsi 2 — model lain
MODEL_PATH = '/content/drive/MyDrive/palm_yolo_output/palm_detection/weights/yolov9_trees.onnx'
# ──────────────────────────────────────────────────────────────────────────────

# ── PARAMETER DETEKSI──────────────────────────────────────
CONFIDENCE  = 0.6
IOU_THRESH  = 0.5
CHIP_SIZE   = 256    
# ──────────────────────────────────────────────────────────────────────────────

# Cek file model ada
import os
assert os.path.exists(MODEL_PATH), f'File model tidak ditemukan: {MODEL_PATH}'

# Load model
model = YOLO(MODEL_PATH)
print(f'Model loaded : {MODEL_PATH}')

# Load raster
raster_path = '/content/drive/MyDrive/sawit_projected.tif'

results_list = []

with rasterio.open(raster_path) as src:
    transform = src.transform
    crs       = src.crs
    width     = src.width
    height    = src.height

    total_chips = ((height // CHIP_SIZE) + 1) * ((width // CHIP_SIZE) + 1)
    processed = 0

    for row in range(0, height, CHIP_SIZE):
        for col in range(0, width, CHIP_SIZE):
            window = Window(col, row,
                            min(CHIP_SIZE, width  - col),
                            min(CHIP_SIZE, height - row))
            chip = src.read([1, 2, 3], window=window)
            chip = np.moveaxis(chip, 0, -1).astype(np.uint8)

            results = model.predict(
                chip,
                conf=CONFIDENCE,
                iou=IOU_THRESH,
                verbose=False
            )

            for result in results:
                boxes = result.boxes.xyxy
                confs = result.boxes.conf

                for i, bbox in enumerate(boxes):
                    x1, y1, x2, y2 = bbox.tolist()
                    geo_x1, geo_y1 = rasterio.transform.xy(transform, row + y1, col + x1)
                    geo_x2, geo_y2 = rasterio.transform.xy(transform, row + y2, col + x2)

                    results_list.append({
                        'geometry'  : box(geo_x1, geo_y2, geo_x2, geo_y1),
                        'confidence': float(confs[i])
                    })

            processed += 1
            if processed % 50 == 0:
                print(f'  Processing: {processed}/{total_chips} chips...', end='\r')

print(f'\nTotal deteksi: {len(results_list)}')

# ── Global NMS — buang duplikat di border chip ─────────────────────────────────
def calc_iou(a, b):
    inter = a.intersection(b).area
    union = a.union(b).area
    return inter / union if union > 0 else 0

def nms_geometries(gdf, iou_threshold):
    gdf = gdf.sort_values('confidence', ascending=False).reset_index(drop=True)
    keep = []
    suppressed = set()
    for i in range(len(gdf)):
        if i in suppressed:
            continue
        keep.append(i)
        for j in range(i + 1, len(gdf)):
            if j in suppressed:
                continue
            if calc_iou(gdf.geometry[i], gdf.geometry[j]) > iou_threshold:
                suppressed.add(j)
    return gdf.iloc[keep].reset_index(drop=True)

gdf = gpd.GeoDataFrame(results_list, crs=crs)
gdf_nms = nms_geometries(gdf, iou_threshold=IOU_THRESH)

print(f'Setelah NMS global   : {len(gdf_nms)}')
print(f'Duplikat dibuang     : {len(gdf) - len(gdf_nms)}')

output_shp = '/content/drive/MyDrive/palm_detection.shp'
gdf_nms.to_file(output_shp)
print(f'Shapefile tersimpan  : {output_shp}')

## Step 11: VARI Health Assessment Percentile

| Kelas | Percentile | Label | Action |
|-------|-----------|-------|--------|
| 1 | > P80 | Healthy | No action |
| 2 | P60 – P80 | Mild Stress | Monitor |
| 3 | P40 – P60 | Moderate Stress | Schedule inspection |
| 4 | P20 – P40 | Severe Stress | Priority inspection |
| 5 | < P20 | Critical | Immediate inspection |

In [ ]:
import numpy as np
import pandas as pd
from rasterio.windows import from_bounds

# ── BAND CONFIG (1-indexed) ────────────────────────────────────────────────────
BAND_R = 1
BAND_G = 2
BAND_B = 3
# ──────────────────────────────────────────────────────────────────────────────

vari_values = []

with rasterio.open(raster_path) as src:
    for _, row_data in gdf_nms.iterrows():
        try:
            win = from_bounds(*row_data.geometry.bounds, src.transform)
            R = src.read(BAND_R, window=win).astype(float).mean()
            G = src.read(BAND_G, window=win).astype(float).mean()
            B = src.read(BAND_B, window=win).astype(float).mean()
            vari = (G - R) / (G + R - B + 1e-10)
        except Exception:
            vari = np.nan
        vari_values.append(vari)

gdf_nms['vari'] = vari_values

# ── Percentile cut-offs ────────────────────────────────────────────────────────
valid = np.array([v for v in vari_values if not np.isnan(v)])
p20 = np.percentile(valid, 20)
p40 = np.percentile(valid, 40)
p60 = np.percentile(valid, 60)
p80 = np.percentile(valid, 80)

print(f'VARI range : {valid.min():.4f} → {valid.max():.4f}')
print(f'P20={p20:.4f}  P40={p40:.4f}  P60={p60:.4f}  P80={p80:.4f}')

# ── Klasifikasi 5 kelas ───────────────────────────────────────────────────────
def classify_vari(vari):
    if np.isnan(vari):  return None
    if vari > p80:      return 'Healthy'
    elif vari > p60:    return 'Mild Stress'
    elif vari > p40:    return 'Moderate Stress'
    elif vari > p20:    return 'Severe Stress'
    else:               return 'Critical'

def get_action(label):
    actions = {
        'Healthy': 'No action',
        'Mild Stress': 'Monitor',
        'Moderate Stress': 'Schedule inspection',
        'Severe Stress': 'Priority inspection',
        'Critical': 'Immediate inspection'
    }
    return actions.get(label, '')

gdf_nms['health'] = gdf_nms['vari'].apply(classify_vari)
gdf_nms['action'] = gdf_nms['health'].apply(get_action)

# ── Export SHP ─────────────────────────────────────────────────────────────────
gdf_nms.to_file('/content/drive/MyDrive/palm_health_vari.shp')
print('Shapefile tersimpan: /content/drive/MyDrive/palm_health_vari.shp')


## Step 12: Summary Report

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
#                    PALM HEALTH ASSESSMENT — SUMMARY
# ══════════════════════════════════════════════════════════════════════════════

total_trees = len(gdf_nms)
kelas_order = ['Healthy', 'Mild Stress', 'Moderate Stress', 'Severe Stress', 'Critical']
action_map  = {
    'Healthy': 'No action',
    'Mild Stress': 'Monitor',
    'Moderate Stress': 'Schedule inspection',
    'Severe Stress': 'Priority inspection',
    'Critical': 'Immediate inspection'
}

summary = (
    gdf_nms.groupby('health')
    .agg(
        jumlah    = ('vari', 'count'),
        vari_min  = ('vari', 'min'),
        vari_max  = ('vari', 'max'),
        vari_mean = ('vari', 'mean'),
    )
    .reindex(kelas_order)
    .round(4)
)
summary['persen_%'] = (summary['jumlah'] / total_trees * 100).round(1)
summary['action']   = summary.index.map(action_map)

# ── Print Report ───────────────────────────────────────────────────────────────
print('='*65)
print('        PALM OIL TREE HEALTH ASSESSMENT REPORT')
print('='*65)
print(f'  Model                     : {os.path.basename(MODEL_PATH)}')
print(f'  Total pohon terdeteksi    : {total_trees}')
print(f'  Confidence / IoU          : {CONFIDENCE} / {IOU_THRESH}')
print(f'  VARI percentile cut-offs  : P20={p20:.4f} P40={p40:.4f} P60={p60:.4f} P80={p80:.4f}')
print('-'*65)
print(f'  {"Kelas":<18} {"Jumlah":>7} {"Persen":>7}   {"Action"}')
print('-'*65)

needs_attention = 0
for kelas in kelas_order:
    row = summary.loc[kelas]
    jumlah = int(row['jumlah'])
    persen = row['persen_%']
    action = row['action']
    print(f'  {kelas:<18} {jumlah:>7} {persen:>6.1f}%   {action}')
    if kelas in ['Moderate Stress', 'Severe Stress', 'Critical']:
        needs_attention += jumlah

print('-'*65)
print(f'  Pohon butuh perhatian     : {needs_attention} ({needs_attention/total_trees*100:.1f}%)')
print(f'  Pohon sehat/monitor saja  : {total_trees - needs_attention} ({(total_trees-needs_attention)/total_trees*100:.1f}%)')
print('='*65)

# ── Export Excel ───────────────────────────────────────────────────────────────
output_xlsx = '/content/drive/MyDrive/palm_health_summary.xlsx'

with pd.ExcelWriter(output_xlsx, engine='openpyxl') as writer:
    summary.to_excel(writer, sheet_name='Summary')
    gdf_nms.drop(columns='geometry').to_excel(writer, sheet_name='Per Pohon', index=False)

print(f'\n  Excel : {output_xlsx}')
print(f'  SHP   : /content/drive/MyDrive/palm_health_vari.shp')
print('\nDone!')